# Assignment 10 - Problem 1
## Turning an English request into SQL with the OpenAI API

**Name:** Hussein Jawad

The request I need to convert to SQL:

> "Select all customers who are born in either June, July, October, live in Boston, are senior citizens and hence qualify for Medicare benefit"

Contents:
- OpenAI account and API key
- Part a - Install the OpenAI library
- Part b - Ask OpenAI to write the SQL
- Part c - Choose the prompt and test temperature and other settings for accuracy
- Part d - The JSON response
- Appendix - Concepts used in this problem

## OpenAI account and API key

I created an account on platform.openai.com, added billing credits, and created a new secret key under **API keys**. I named the key `e89-hw10-key`. The key is stored in a `.env` file in the same folder as this notebook, not in the notebook itself:

```
OPENAI_API_KEY=sk-...
OPENAI_KEY_NAME=e89-hw10-key
```

<!-- Paste the screenshot of the API keys page here (in Colab you can drag the image into this cell) -->

## Part a - Install the OpenAI library and set up the client

In [1]:
%pip install -q --upgrade openai python-dotenv

I run the notebook in Google Colab, and the notebook and `.env` file are in my Google Drive folder. Colab starts in `/content`, so I mount Drive and change into that folder first. When the notebook runs locally, this cell does nothing.

In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10"   # folder in Google Drive with this notebook and .env

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT_DIR)
except ImportError:
    pass   # not running in Colab

print("Working folder:", os.getcwd())
print(".env found    :", os.path.exists(".env"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10
.env found    : True


Load the key from `.env` and create the client. If OpenAI returns `model_not_found`, change `MODEL` / `SMALL_MODEL`.

In [4]:
import json
import openai
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True), override=True)

MODEL = "gpt-5.4"
SMALL_MODEL = "gpt-5.4-mini"
client = OpenAI()   # reads OPENAI_API_KEY from the environment

key = os.getenv("OPENAI_API_KEY")
assert key, "OPENAI_API_KEY not found - check the .env file"
print("openai version:", openai.__version__)
print("Key name      :", os.getenv("OPENAI_KEY_NAME"))
print("Key           :", key[:3] + "..." + key[-4:])

def show_json(obj):
    display(json.loads(obj.model_dump_json()))

openai version: 3.52.0
Key name      : e89-hw10-key
Key           : sk-...iIoA


## Part b - Ask OpenAI to write the SQL

The assignment doesn't give a table, so I made a `Customers` table. It needs a `DateOfBirth` column to get the birth month and the age. I also added an `Orders` table that the query should not need. Being a senior citizen who qualifies for Medicare means being 65 or older.

In [5]:
SCHEMA = '''
CREATE TABLE Customers (
  CustomerID   INTEGER PRIMARY KEY,
  FirstName    VARCHAR(50),
  LastName     VARCHAR(50),
  DateOfBirth  DATE,          -- 'YYYY-MM-DD'
  Email        VARCHAR(100),
  Phone        VARCHAR(20),
  City         VARCHAR(50),
  State        CHAR(2)        -- e.g. 'MA'
);

CREATE TABLE Orders (
  OrderID      INTEGER PRIMARY KEY,
  CustomerID   INTEGER REFERENCES Customers(CustomerID),
  OrderDate    DATE,
  TotalAmount  DECIMAL(10, 2)
);
'''.strip()

REQUEST = ("Select all customers who are born in either June, July, October, live in Boston, "
           "are senior citizens and hence qualify for Medicare benefit")

First I used the same kind of prompt as the SQL example in the lab: the tables go in the system message and the request in the user message.

In [6]:
baseline_messages = [
    {"role": "system",
     "content": "Given the following SQL tables, your job is to write queries given a user's request.\n\n" + SCHEMA},
    {"role": "user", "content": REQUEST},
]

baseline_response = client.responses.create(
    model=MODEL,
    input=baseline_messages,
    reasoning={"effort": "none"},   # needed to be able to set temperature
    temperature=0,
    max_output_tokens=1000,
)
print(baseline_response.output_text)

```sql
SELECT *
FROM Customers
WHERE City = 'Boston'
  AND EXTRACT(MONTH FROM DateOfBirth) IN (6, 7, 10)
  AND DateOfBirth <= CURRENT_DATE - INTERVAL '65 years';
```


The model wrote reasonable SQL, but there are problems:
- It's plain text with the SQL inside a code block, so a program can't use it directly.
- The prompt doesn't say which database it is for, so the model used PostgreSQL syntax (`EXTRACT(MONTH FROM ...)`, `CURRENT_DATE - INTERVAL '65 years'`).
- It only checks the city, not the state.

So in part c I improved the prompt and asked for the answer as JSON.

## Part c - Choosing the prompt and testing settings for accuracy

### c.1 A better prompt

What I changed compared to the first prompt:
- The request goes between `<request>` tags and the tables between `<schema>` tags (delimiters from Notebook 1).
- I ask for SQLite, so I can run the query inside this notebook, and tell it not to hard-code today's date.
- It has to list the conditions it found before it writes the SQL, so it works the problem out first.
- The answer has to be JSON. Instead of only asking for JSON in the prompt, I used Structured Outputs (`text.format` with a JSON schema), so the reply always parses.

In [7]:
SYSTEM_PROMPT = f'''
You are an expert SQL developer. Translate the request delimited by <request></request>
into one SQL query for this database:

<schema>
{SCHEMA}
</schema>

Rules:
- Use SQLite syntax and only the tables and columns above.
- Write a single read-only SELECT statement.
- Dates are text 'YYYY-MM-DD'. Use date('now') for today, never a hard-coded date.
- If a condition depends on age, compute the exact age.
- "Boston" means City = 'Boston' and State = 'MA'.
- "Select all customers" means return all columns.
- First list the conditions you found in the request, then write the query.
- If the request can't be answered with these tables, set "sql" to null and explain why.
'''.strip()

SQL_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "conditions":  {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
        "sql":         {"type": ["string", "null"]},
        "explanation": {"type": "string"},
    },
    "required": ["conditions", "assumptions", "sql", "explanation"],
    "additionalProperties": False,
}
TEXT_FORMAT = {"format": {"type": "json_schema", "name": "nl_to_sql",
                          "schema": SQL_JSON_SCHEMA, "strict": True}}

I use one function for all calls so I can change the prompt, the model and the settings easily. Following Notebook 1, temperature and top_p are only allowed when reasoning effort is `"none"`, so the function leaves them out for other effort levels.

In [8]:
def generate_sql(prompt="engineered", model=MODEL, temperature=None, top_p=None, effort="none"):
    if prompt == "baseline":
        args = {"input": baseline_messages}
    else:
        args = {"input": [{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": f"<request>{REQUEST}</request>"}],
                "text": TEXT_FORMAT}
    if effort == "none":
        if temperature is not None: args["temperature"] = temperature
        if top_p is not None:       args["top_p"] = top_p
    max_tokens = 1000 if effort == "none" else 4000   # reasoning tokens count too
    return client.responses.create(model=model, reasoning={"effort": effort},
                                   max_output_tokens=max_tokens, **args)

response = generate_sql("engineered", MODEL, temperature=0)
result = json.loads(response.output_text)
print("status:", response.status)
print(json.dumps(result, indent=2))
print("\nSQL:\n" + result["sql"])

status: completed
{
  "conditions": [
    "Return all columns from Customers",
    "Customer lives in Boston (City = 'Boston' and State = 'MA')",
    "Customer was born in June, July, or October",
    "Customer is a senior citizen and qualifies for Medicare benefit, interpreted as age 65 or older",
    "Age must be computed exactly using today's date"
  ],
  "assumptions": [
    "\"Senior citizens and hence qualify for Medicare benefit\" is interpreted as customers who are at least 65 years old",
    "Month of birth is determined from DateOfBirth using SQLite strftime('%m', DateOfBirth)"
  ],
  "sql": "SELECT *\nFROM Customers\nWHERE City = 'Boston'\n  AND State = 'MA'\n  AND strftime('%m', DateOfBirth) IN ('06', '07', '10')\n  AND (\n    CAST(strftime('%Y', date('now')) AS INTEGER)\n    - CAST(strftime('%Y', DateOfBirth) AS INTEGER)\n    - (strftime('%m-%d', date('now')) < strftime('%m-%d', DateOfBirth))\n  ) >= 65;",
  "explanation": "The query returns all customer columns for people

This time the model listed the conditions first and worked out on its own that "qualify for Medicare" means age 65 or older. It computes the exact age: the difference in birth years, minus 1 if the birthday hasn't happened yet this year.

### c.2 How I measure accuracy

A query can look right and still be wrong, so I tested it. I made a small SQLite database where each customer tests one condition (wrong month, wrong city, too young, exactly 65, missing birth date). I worked out the correct answer in Python and compared it with what the query returns. The birth dates are based on today's date, so the test still works if I run it later.

In [9]:
import re, sqlite3
import pandas as pd
from datetime import date, datetime, timezone

TODAY = datetime.now(timezone.utc).date()   # SQLite's date('now') uses UTC

def born(years_ago, month, day):
    return date(TODAY.year - years_ago, month, day).isoformat()

# (id, first, last, date of birth, city, state)   what the row tests
CUSTOMERS = [
    (1,  "Mary",  "Smith",  born(72, 6, 15),  "Boston",    "MA"),   # match (June)
    (2,  "John",  "Brown",  born(80, 7, 4),   "Boston",    "MA"),   # match (July)
    (3,  "Ann",   "Lee",    born(66, 10, 31), "Boston",    "MA"),   # match (October)
    (4,  "Eve",   "Clark",  born(65, 10, 1),  "Boston",    "MA"),   # exactly 65 (>= vs >)
    (5,  "Lena",  "Wong",   born(85, 6, 30),  "Boston",    "MA"),   # match
    (6,  "Bob",   "Jones",  born(75, 5, 10),  "Boston",    "MA"),   # wrong month
    (7,  "Grace", "Hill",   born(90, 12, 1),  "Boston",    "MA"),   # wrong month
    (8,  "Carl",  "Davis",  born(70, 6, 1),   "Cambridge", "MA"),   # wrong city
    (9,  "Dana",  "White",  born(40, 6, 20),  "Boston",    "MA"),   # too young
    (10, "Frank", "Green",  born(64, 7, 15),  "Boston",    "MA"),   # 63 or 64, too young
    (11, "Omar",  "King",   None,             "Boston",    "MA"),   # no birth date
]

def build_test_db():
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    con.executemany("INSERT INTO Customers VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                    [(i, f, l, dob, f.lower() + "@example.com", "617-555-01%02d" % i, city, st)
                     for i, f, l, dob, city, st in CUSTOMERS])
    con.execute("INSERT INTO Orders VALUES (1, 1, '2026-01-05', 120.50)")
    con.execute("PRAGMA query_only = ON")   # the generated SQL can't change anything
    return con

def age(dob):
    d = date.fromisoformat(dob)
    return TODAY.year - d.year - ((TODAY.month, TODAY.day) < (d.month, d.day))

# correct answer, computed in Python
EXPECTED = {i for i, f, l, dob, city, st in CUSTOMERS
            if dob and date.fromisoformat(dob).month in (6, 7, 10)
            and city == "Boston" and st == "MA" and age(dob) >= 65}
print("Today:", TODAY, "- expected CustomerIDs:", sorted(EXPECTED))

Today: 2026-11-17 - expected CustomerIDs: [1, 2, 3, 4, 5]


`check_sql` runs a query on the test database and compares the returned IDs with the expected ones. For the first prompt it takes the SQL out of the code block, and it refuses anything that isn't a SELECT.

In [10]:
def extract_sql(text):
    m = re.search(r"```(?:sql|sqlite)?\s*(.*?)```", text, re.S | re.I)
    return (m.group(1) if m else text).strip()

def check_sql(sql):
    out = {"runs": False, "correct": False, "ids": None, "error": None}
    if not sql or not re.match(r"^\s*(WITH|SELECT)\b", sql, re.I):
        out["error"] = "no SELECT statement"
        return out
    try:
        cur = build_test_db().execute(sql)
        cols = [c[0].lower() for c in cur.description]
        ids = {row[cols.index("customerid")] for row in cur.fetchall()}
        out.update(runs=True, ids=sorted(ids), correct=(ids == EXPECTED))
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out

baseline_sql = extract_sql(baseline_response.output_text)
print("First prompt :", check_sql(baseline_sql))
print("Better prompt:", check_sql(result["sql"]))

First prompt : {'runs': False, 'correct': False, 'ids': None, 'error': 'OperationalError: near "FROM": syntax error'}
Better prompt: {'runs': True, 'correct': True, 'ids': [1, 2, 3, 4, 5], 'error': None}


### c.3 Trying different settings

From Notebook 1: temperature controls how random the output is (0 = almost always the same), top_p limits the choice to the most likely tokens, and reasoning effort lets the model think first (temperature and top_p can't be used then). I also compared the smaller model. I ran every setting 3 times, checked each query with `check_sql`, and counted how many different queries each setting gave.

In [11]:
import time

SETTINGS = [
    ("first prompt, T=0",      "baseline",   MODEL,       {"temperature": 0}),
    ("better prompt, T=0",     "engineered", MODEL,       {"temperature": 0}),
    ("better prompt, T=0.7",   "engineered", MODEL,       {"temperature": 0.7}),
    ("better prompt, T=1.5",   "engineered", MODEL,       {"temperature": 1.5}),
    ("better prompt, top_p=0.1", "engineered", MODEL,     {"top_p": 0.1}),
    ("better prompt, mini, T=0", "engineered", SMALL_MODEL, {"temperature": 0}),
    ("better prompt, effort=low", "engineered", MODEL,    {"effort": "low"}),
]
N_RUNS = 3

rows = []
for label, prompt, model, settings in SETTINGS:
    for run in range(N_RUNS):
        start = time.time()
        try:
            r = generate_sql(prompt, model, **settings)
            sql = extract_sql(r.output_text) if prompt == "baseline" else json.loads(r.output_text)["sql"]
            rows.append({"setting": label, "sql": sql, "tokens": r.usage.total_tokens,
                         "seconds": time.time() - start, **check_sql(sql)})
        except Exception as e:   # e.g. a setting the model doesn't support
            rows.append({"setting": label, "sql": None, "runs": False, "correct": False,
                         "error": f"{type(e).__name__}: {e}"[:150]})
    print("done:", label)

runs = pd.DataFrame(rows)
summary = runs.groupby("setting", sort=False).agg(
    accuracy=("correct", "mean"),
    runs_on_sqlite=("runs", "mean"),
    different_queries=("sql", lambda s: s.dropna().str.strip().nunique()),
    avg_tokens=("tokens", "mean"),
    avg_seconds=("seconds", "mean"),
).round(2)
summary

done: first prompt, T=0
done: better prompt, T=0
done: better prompt, T=0.7
done: better prompt, T=1.5
done: better prompt, top_p=0.1
done: better prompt, mini, T=0
done: better prompt, effort=low


,accuracy,runs_on_sqlite,different_queries,avg_tokens,avg_seconds
setting,,,,,
"first prompt, T=0",0.0,0.0,2,239.67,2.69
"better prompt, T=0",1.0,1.0,1,660.00,5.26
"better prompt, T=0.7",1.0,1.0,2,676.00,6.41
"better prompt, T=1.5",1.0,1.0,3,668.33,7.89
"better prompt, top_p=0.1",1.0,1.0,1,660.00,5.24
"better prompt, mini, T=0",1.0,1.0,2,643.67,3.13
"better prompt, effort=low",1.0,1.0,2,738.67,6.13


### Observations

- **The prompt mattered the most.** The first prompt got 0 out of 3 right. Its logic was fine, but it was written for PostgreSQL (`EXTRACT(MONTH FROM DateOfBirth)`, `INTERVAL '65 years'`), so SQLite gave `near "FROM": syntax error` every time. The better prompt names the database and the date format, and it was correct in all 18 of its runs, at every setting.
- **JSON.** With Structured Outputs, every run of the better prompt returned JSON that parsed, even at temperature 1.5.
- **Temperature.** Higher temperature gave more variety: 1 different query in 3 runs at T=0, 2 at T=0.7 and 3 at T=1.5. All of them were still correct, so only the way the query was written changed, not the logic. Temperature 0 wasn't perfectly repeatable either: the first prompt and the mini model each gave 2 different queries at T=0.
- **top_p = 0.1** acted like temperature 0 and gave the same query in all 3 runs.
- **Reasoning effort "low"** used more tokens (739 vs 660 on average) and took longer (6.1 s vs 5.3 s) with no gain in accuracy, because the task is simple once the prompt is clear.
- **The smaller model** (gpt-5.4-mini) was also 100% correct and the fastest (3.1 s vs 5.3 s), but less consistent.
- **Cost.** The better prompt uses more tokens than the first one (about 660 vs 240) because of the longer instructions and the JSON fields, but that is still a very small request.

For part d I used the better prompt with `gpt-5.4`, temperature 0 and no reasoning: it was always correct, gave the same query every time and used no extra reasoning tokens.

## Part d - The JSON response

The final call uses the settings I picked in part c. `show_json` prints the whole API response, which is itself JSON.

In [13]:
final_response = generate_sql("engineered", MODEL, temperature=0)
show_json(final_response)

The output of the cell above was very long, so I deleted it and copied the important fields here. The model's answer is the text in `output[0].content[0].text` and is shown in full in the next cell.

```json
{
  "id": "resp_03443d6f27e97f01006afbe6d8638487d2b2da65a2794a474c",
  "object": "response",
  "model": "gpt-5.4-2026-03-05",
  "status": "completed",
  "error": null,
  "incomplete_details": null,
  "output": [
    {
      "type": "message",
      "role": "assistant",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "text": "{\"conditions\":[...],\"assumptions\":[...],\"sql\":\"SELECT * FROM Customers WHERE ...\",\"explanation\":\"...\"}"
        }
      ]
    }
  ],
  "temperature": 0.0,
  "top_p": 0.98,
  "reasoning": {
    "effort": "none"
  },
  "max_output_tokens": 1000,
  "text": {
    "format": {
      "type": "json_schema",
      "name": "nl_to_sql",
      "strict": true
    }
  },
  "usage": {
    "input_tokens": 378,
    "output_tokens": 282,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 660
  }
}
```

The model's answer parsed from that text, together with the request, the settings, the token usage and my accuracy check:

In [14]:
final_answer = json.loads(final_response.output_text)

final = {
    "name": "Hussein Jawad",
    "request": REQUEST,
    "response_id": final_response.id,
    "model": final_response.model,
    "status": final_response.status,
    "settings": {"temperature": 0, "reasoning_effort": "none", "max_output_tokens": 1000,
                 "output_format": "json_schema (strict)"},
    "usage": {"input_tokens": final_response.usage.input_tokens,
              "output_tokens": final_response.usage.output_tokens,
              "total_tokens": final_response.usage.total_tokens},
    "answer": final_answer,
    "accuracy_check": check_sql(final_answer["sql"]),
}
print(json.dumps(final, indent=2))
print("\nFinal SQL:\n" + final_answer["sql"])

{
  "name": "Hussein Jawad",
  "request": "Select all customers who are born in either June, July, October, live in Boston, are senior citizens and hence qualify for Medicare benefit",
  "response_id": "resp_03443d6f27e97f01006afbe6d8638487d2b2da65a2794a474c",
  "model": "gpt-5.4-2026-03-05",
  "status": "completed",
  "settings": {
    "temperature": 0,
    "reasoning_effort": "none",
    "max_output_tokens": 1000,
    "output_format": "json_schema (strict)"
  },
  "usage": {
    "input_tokens": 378,
    "output_tokens": 282,
    "total_tokens": 660
  },
  "answer": {
    "conditions": [
      "Return all columns from Customers.",
      "Customer lives in Boston (City = 'Boston' and State = 'MA').",
      "Customer was born in June, July, or October.",
      "Customer is a senior citizen, interpreted as age 65 or older as of today.",
      "Use exact age calculation."
    ],
    "assumptions": [
      "\"Qualify for Medicare benefit\" is interpreted as being age 65 or older, since no 

## Appendix - Concepts used in this problem

**Large language model (LLM).** A neural network (a transformer) trained on a huge amount of text to predict the next token. It writes an answer one token at a time. GPT-5.4 is the LLM I used here.

**Token.** A piece of text, usually part of a word. Requests are measured and billed in tokens. The response's `usage` field shows `input_tokens` (my prompt), `output_tokens` (the answer, including any reasoning tokens) and `total_tokens`. `max_output_tokens` sets the most the model may write.

**API key.** A secret password that identifies my OpenAI account and is used for billing. It is kept in a `.env` file, loaded with `python-dotenv`, and never written in the notebook or uploaded to GitHub.

**OpenAI Responses API.** `client.responses.create(model=..., input=...)` sends a request and returns a response object. The useful parts are `output_text` (the answer), `status` (`completed` or `incomplete`) and `usage` (tokens).

**Message roles.** The input is a list of messages. The **system** message sets the rules and context (here: the table definitions and instructions), and the **user** message is the actual request.

**Prompt engineering.** Writing the prompt so the model does what you want. From Notebook 1:
- *Clear and specific instructions*: use delimiters (tags) to separate the parts of the input, ask for a structured output, and say what to do when the task can't be done.
- *Give the model time to think*: ask it to work out the steps (here: the list of conditions) before the final answer.

**Structured Outputs.** With `text={"format": {"type": "json_schema", "strict": True, ...}}` the API forces the answer to be valid JSON that matches my schema (fields `conditions`, `assumptions`, `sql`, `explanation`). Only asking for JSON in the prompt usually works, but it isn't guaranteed.

**Temperature.** Controls how random the choice of the next token is. The model gives each possible token a score z, and the probability of token i is `p_i = exp(z_i / T) / sum_j exp(z_j / T)`. A low T almost always picks the most likely token (consistent output). A high T spreads the probability out (more variety, more risk). Consistent doesn't mean correct.

**top_p (nucleus sampling).** The model samples only from the smallest set of tokens whose probabilities add up to p. With `top_p=0.1` only the most likely tokens are left, so it behaves like a low temperature. OpenAI recommends changing temperature or top_p, not both.

**Reasoning effort.** Reasoning models can "think" (use hidden reasoning tokens) before answering. `effort` can be `none`, `low`, `medium` or `high`. More effort costs more tokens and time. Temperature and top_p can only be set when effort is `none`.

**Text-to-SQL and SQL dialects.** The model turns an English request into SQL, using the table definitions I gave it. Databases use different syntax for the same idea, so the prompt has to name the database:

| Idea | SQLite | PostgreSQL | MySQL |
|---|---|---|---|
| birth month | `strftime('%m', DateOfBirth)` | `EXTRACT(MONTH FROM DateOfBirth)` | `MONTH(DateOfBirth)` |
| today | `date('now')` | `CURRENT_DATE` | `CURDATE()` |

**Medicare and exact age.** Medicare starts at age 65, so "senior citizen who qualifies for Medicare" means age >= 65. The exact age is (this year - birth year), minus 1 if the birthday hasn't happened yet this year. Using only the difference in years would count some 64-year-olds as 65.

**Checking generated SQL.** An LLM can write SQL that looks right but is wrong or doesn't run. I tested every query on a small database where I knew the correct answer, ran it read-only (`PRAGMA query_only = ON`), and only accepted it if it returned exactly the right customers.